# Tiling Large Datasets

Split point clouds into spatial tiles for processing datasets that exceed RAM.

**Modules used:** `occulus.tiling`, `occulus.segmentation`

In [ ]:
import numpy as np
from occulus.types import PointCloud
from occulus.tiling import tile_point_cloud, iter_tiles, Tile

## Create a synthetic large cloud

In [ ]:
rng = np.random.default_rng(42)
xyz = np.column_stack([
    rng.uniform(0, 1000, 100_000),
    rng.uniform(0, 1000, 100_000),
    rng.uniform(0, 50, 100_000),
])
cloud = PointCloud(xyz)
print(f"Total points: {cloud.n_points:,}")

## Tile into 250m chunks

In [ ]:
import tempfile
from pathlib import Path

out_dir = Path(tempfile.mkdtemp()) / "tiles"
tiles = tile_point_cloud(cloud, out_dir, tile_size=250)

print(f"Created {len(tiles)} tiles")
for t in tiles[:5]:
    print(f"  Tile {t.index}: {t.point_count:,} pts, bounds={t.bounds}")

## Iterate lazily over tiles

In [ ]:
# Write cloud to temp file first
from occulus.io import write
tmp_file = Path(tempfile.mkdtemp()) / "cloud.xyz"
write(cloud, tmp_file)

for tile, tile_cloud in iter_tiles(str(tmp_file), tile_size=500):
    print(f"Tile {tile.index}: {tile_cloud.n_points:,} points")